# 04. Evaluación complementaria: sanas del repositorio que no se usaron

El conjunto analizado incluye 1268 de las 3344 imágenes sanas del repositorio. Este notebook evalúa los 75 modelos ya entrenados en `03_entrenamiento_validacion_cruzada.ipynb` (5 arquitecturas × 3 repeticiones × 5 pliegues) sobre las sanas que quedaron fuera, sin reentrenar.

1. Identifica las sanas no usadas por MD5 y las prepara igual que en el entrenamiento (RGB, copia de 256 px bicúbica).
2. Excluye las que la auditoría de identidad (notebooks 01 y 02) vincula con el conjunto principal; marca además los vínculos dudosos para un análisis de sensibilidad y la escena «mazorca en la mano sobre un recipiente».
3. Predice con cada modelo y, como control, repite la predicción de su propio pliegue de prueba para confirmar que coincide con la guardada.
4. Calcula especificidad (umbral 0.5 y umbral de validación), falsos positivos, especificidad por escena, AUC con negativos nuevos (BPR del pliegue de prueba frente a sanas no usadas) y la diferencia de especificidad respecto de las sanas usadas, con IC95 % por bootstrap de grupos.

Requisitos: `dataset_complete/cacao_diseases/cacao_photos/`, `resultados/` con los puntos de control (`resultados/ckpt/`, no incluidos en el repositorio por su tamaño) y `auditoria_identidad/`. Las salidas van a `resultados_sanas_no_usadas/`.


## 1. Configuración

In [ ]:
import os, sys, json, glob, time, hashlib
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import torch
import torch.nn as nn
import timm
import imagehash
import torchvision.transforms as T
from timm.data import resolve_data_config
from sklearn.metrics import roc_auc_score
from IPython.display import display

# ===================== CONFIGURACIÓN =====================
PROJECT_DIR = Path.cwd()                                   # raíz del repositorio
REPO_DIR = PROJECT_DIR / 'dataset_complete' / 'cacao_diseases' / 'cacao_photos'
RES = PROJECT_DIR / 'resultados'                           # resultados de la validación cruzada (solo lectura)
A4 = PROJECT_DIR / 'auditoria_identidad'                   # auditoría de identidad (notebooks 01 y 02)
OUT = PROJECT_DIR / 'resultados_sanas_no_usadas'          # salidas de esta evaluación complementaria
CACHE_DIR = PROJECT_DIR / 'cache_256'
LIMITE_IMG = None                                          # None = todas; un número solo para pruebas rápidas
LIMITE_CKPT = None                                         # None = los 15 modelos por arquitectura
BATCH = 256

for d in ('tablas', 'figs', 'auditoria'):
    (OUT / d).mkdir(parents=True, exist_ok=True)
CFG = json.load(open(RES / 'config.json'))
MODELOS = ['SwinT', 'ConvNeXtT', 'DeiT', 'EfficientNetV2S', 'ResNet50']
DEV = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__, '| timm', timm.__version__, '| dispositivo', DEV,
      torch.cuda.get_device_name(0) if DEV.type == 'cuda' else '')

def contenedor_a_local(p):
    # Rutas de folds.csv relativas a la raíz (o del contenedor, /workspace/cacao/)
    p = str(p)
    return str(PROJECT_DIR / p.split('/workspace/cacao/', 1)[1]) if '/workspace/cacao/' in p else p

folds = pd.read_csv(RES / 'folds.csv')
folds['cache_path'] = folds['cache_path'].apply(contenedor_a_local)
print('Imágenes del experimento principal:', len(folds), '|', folds['label'].value_counts().rename({0: 'Sana', 1: 'BPR'}).to_dict())


## 2. Sanas no usadas y preprocesamiento

In [ ]:
# 1) Imágenes sanas del repositorio completo que NO se usaron en el experimento principal.
#    La correspondencia se hace por MD5 (contenido del archivo), no por nombre.
sanas_repo = sorted(glob.glob(str(REPO_DIR / 'healthy' / '*.jpg')))
md5_usadas = set(folds['md5'])

def md5(p):
    with open(p, 'rb') as fh:
        return hashlib.md5(fh.read()).hexdigest()

with ThreadPoolExecutor(16) as ex:
    md5_repo = list(ex.map(md5, sanas_repo))
nu = pd.DataFrame({'abs_path': sanas_repo, 'md5': md5_repo})
nu = nu[~nu['md5'].isin(md5_usadas)].reset_index(drop=True)
assert nu['md5'].nunique() == len(nu)
print('Sanas en el repositorio: %d | usadas en el experimento: %d | no usadas: %d' % (
    len(sanas_repo), int((folds['label'] == 0).sum()), len(nu)))
if LIMITE_IMG:
    nu = nu.sample(LIMITE_IMG, random_state=0).reset_index(drop=True)
    print('PRUEBA: se limitan a', len(nu))
nu['image_path'] = nu['abs_path'].apply(lambda p: str(Path(p).relative_to(PROJECT_DIR)))
nu['label'] = 0

# 2) Mismo preprocesamiento que el experimento principal: RGB y copia de 256 px (bicúbica)
def procesar(row):
    dst = CACHE_DIR / (os.path.splitext(row.image_path)[0] + '.png')
    dst.parent.mkdir(parents=True, exist_ok=True)
    img = Image.open(row.abs_path).convert('RGB')
    w, h = img.size
    ph = str(imagehash.phash(img))
    if not dst.exists():
        img.resize((CFG['cache_size'], CFG['cache_size']), Image.Resampling.BICUBIC).save(dst)
    return ph, w, h, str(dst)

t0 = time.time()
with ThreadPoolExecutor(16) as ex:
    res = list(ex.map(procesar, nu.itertuples()))
nu['phash'], nu['w'], nu['h'], nu['cache_path'] = zip(*res)
tam_fuente = folds.groupby('fuente')['tamano'].first().to_dict()          # {'A': '1080x1080', 'B': '2160x2160'}
inv = {v: k for k, v in tam_fuente.items()}
nu['tamano'] = nu['w'].astype(str) + 'x' + nu['h'].astype(str)
nu['fuente'] = nu['tamano'].map(inv).fillna('otra')
print('Copias de 256 px listas en %.0f s' % (time.time() - t0))
tab = pd.DataFrame({'Usadas (experimento principal)': folds[folds['label'] == 0]['fuente'].value_counts(),
                    'No usadas': nu['fuente'].value_counts()}).fillna(0).astype(int)
tab.loc['Total'] = tab.sum()
display(tab)
tab.to_csv(OUT / 'auditoria' / 'sanas_por_fuente.csv')


## 3. Fotos de la misma mazorca y vínculos con el conjunto principal

In [ ]:
# 3) Fotos de las mismas mazorcas entre las no usadas y las 2211 del conjunto principal
#    (pHash <= 6 o coseno DINOv2 >= 0.95); se excluyen del análisis principal
def ham_matrix(h1, h2):
    a = np.array([int(h, 16) for h in h1], dtype=np.uint64)
    b = np.array([int(h, 16) for h in h2], dtype=np.uint64)
    out = np.zeros((len(a), len(b)), dtype=np.uint8)
    for i in range(0, len(a), 256):
        x = np.bitwise_xor(a[i:i + 256, None], b[None, :])
        out[i:i + 256] = np.unpackbits(x.view(np.uint8), axis=-1).reshape(x.shape[0], len(b), 64).sum(-1)
    return out

fm = timm.create_model(CFG['dedup']['feat_models'][0], pretrained=True, num_classes=0, img_size=224).eval().to(DEV)
dc = resolve_data_config({}, model=fm)
tf_feat = T.Compose([T.Resize((224, 224)), T.ToTensor(), T.Normalize(dc['mean'], dc['std'])])

@torch.no_grad()
def features(paths):
    fs = []
    for i in range(0, len(paths), 128):
        xb = torch.stack([tf_feat(Image.open(p).convert('RGB')) for p in paths[i:i + 128]]).to(DEV)
        fs.append(nn.functional.normalize(fm(xb).float(), dim=1).cpu())
        print('\rCaracterísticas DINOv2: %d/%d' % (min(i + 128, len(paths)), len(paths)), end='', flush=True)
    print()
    return torch.cat(fs).numpy()

f_nu = features(nu['cache_path'].tolist())
f_us = features(folds['cache_path'].tolist())
del fm
if DEV.type == 'cuda':
    torch.cuda.empty_cache()

ham_x = ham_matrix(nu['phash'], folds['phash'])
cos_x = f_nu @ f_us.T
nu['ham_min_usadas'] = ham_x.min(1)
nu['cos_max_usadas'] = cos_x.max(1)
nu['vecino_usado'] = folds['image_path'].values[cos_x.argmax(1)]
nu['clase_vecino'] = folds['label'].values[cos_x.argmax(1)]
nu['gemela'] = (nu['ham_min_usadas'] <= CFG['dedup']['hamming']) | (nu['cos_max_usadas'] >= CFG['dedup']['cos'])
# Vínculos con el conjunto principal según la auditoría de identidad (regla de secuencia de captura y franja dudosa)
vinc = pd.read_csv(A4 / 'sanas_no_usadas_vinculos.csv')[['image_path', 'vinculo_regla', 'vinculo_dudoso']]
nu = nu.merge(vinc, on='image_path', how='left')
nu[['vinculo_regla', 'vinculo_dudoso']] = nu[['vinculo_regla', 'vinculo_dudoso']].fillna(False).astype(bool)
nu['gemela'] = nu['gemela'] | nu['vinculo_regla']          # excluidas del análisis principal
cen = pd.read_csv(A4 / 'censo_escena_cajon.csv')[['image_path', 'sim_escena_cajon']]
nu = nu.merge(cen, on='image_path', how='left')
nu['escena_cajon'] = nu['sim_escena_cajon'] >= 0.80        # mazorca en la mano sobre un recipiente (censo de la auditoría)

# Grupos dentro de las no usadas (para el bootstrap por grupos): mismo criterio de unión
ham_i = ham_matrix(nu['phash'], nu['phash'])
cos_i = f_nu @ f_nu.T
np.fill_diagonal(ham_i, 255)
np.fill_diagonal(cos_i, -1)
padre = list(range(len(nu)))
def raiz(i):
    while padre[i] != i:
        padre[i] = padre[padre[i]]
        i = padre[i]
    return i
iu = np.triu_indices(len(nu), 1)
cand = (ham_i[iu] <= CFG['dedup']['hamming']) | (cos_i[iu] >= CFG['dedup']['cos'])
for a, b in zip(iu[0][cand], iu[1][cand]):
    ra, rb = raiz(a), raiz(b)
    if ra != rb:
        padre[rb] = ra
nu['group'] = [raiz(i) for i in range(len(nu))]

# Referencia: vecino más cercano de cada imagen sana usada respecto de las otras usadas
cos_uu = f_us[folds['label'].values == 0] @ f_us.T
idx_s = np.where(folds['label'].values == 0)[0]
cos_uu[np.arange(len(idx_s)), idx_s] = -1

print('No usadas excluidas (gemela o vínculo de secuencia con el conjunto principal): %d de %d (%.1f %%) | vínculo dudoso adicional: %d | escena mano + recipiente: %d' % (
    nu['gemela'].sum(), len(nu), 100 * nu['gemela'].mean(), int((nu['vinculo_dudoso'] & ~nu['gemela']).sum()), int(nu['escena_cajon'].sum())))
print('Pares entre no usadas que se agrupan: %d | grupos: %d' % (int(cand.sum()), nu['group'].nunique()))
print('\nSimilitud coseno con la imagen más parecida del experimento (percentiles):')
ref = pd.DataFrame({'No usadas -> experimento': pd.Series(nu['cos_max_usadas']).describe(percentiles=[.05, .5, .95, .99]),
                    'Sanas usadas -> resto del experimento': pd.Series(cos_uu.max(1)).describe(percentiles=[.05, .5, .95, .99])})
display(ref.round(3))
ref.to_csv(OUT / 'auditoria' / 'similitud_vecino.csv')
nu.drop(columns=['abs_path']).to_csv(OUT / 'auditoria' / 'sanas_no_usadas.csv', index=False)

# Pares más parecidos para revisión visual
top = nu.sort_values('cos_max_usadas', ascending=False).head(8)
fig, axes = plt.subplots(len(top), 2, figsize=(4.6, 2.3 * len(top)))
for r, (_, t) in enumerate(top.iterrows()):
    vec = folds.set_index('image_path').loc[t['vecino_usado']]
    for c, (p, tit) in enumerate([(t['cache_path'], 'No usada: ' + os.path.basename(t['image_path'])),
                                  (vec['cache_path'], 'Usada: ' + os.path.basename(t['vecino_usado']))]):
        axes[r, c].imshow(Image.open(p)); axes[r, c].set_title(tit, fontsize=6); axes[r, c].axis('off')
    axes[r, 0].text(-0.05, 0.5, 'cos=%.3f\nd=%d\n%s' % (t['cos_max_usadas'], t['ham_min_usadas'], 'GEMELA' if t['gemela'] else ''),
                    transform=axes[r, 0].transAxes, ha='right', va='center', fontsize=6)
plt.tight_layout(); fig.savefig(OUT / 'auditoria' / 'pares_no_usadas_vs_usadas.png', dpi=150); plt.show()


## 4. Inferencia con los 75 modelos

In [ ]:
# 4) Inferencia con los 75 modelos ya entrenados (5 arquitecturas × 3 repeticiones × 5 pliegues), sin reentrenar.
#    Como control, cada modelo también vuelve a predecir su propio pliegue de prueba y se compara con la
#    probabilidad guardada en la validación cruzada: si el preprocesamiento coincide, la diferencia debe ser ~0.
preds_cv = pd.concat([pd.read_csv(p) for p in sorted(glob.glob(str(RES / 'preds' / '*_r?_f?.csv')))], ignore_index=True)
meta_cv = pd.DataFrame([json.load(open(p)) for p in sorted(glob.glob(str(RES / 'preds' / '*_r?_f?.json')))])
cache_de = folds.set_index('image_path')['cache_path']

def tensores(paths, tf):
    return torch.stack([tf(Image.open(p).convert('RGB')) for p in paths])

@torch.no_grad()
def predecir(model, X):
    out = []
    for i in range(0, len(X), BATCH):
        x = X[i:i + BATCH].to(DEV).to(memory_format=torch.channels_last)
        out.append(torch.sigmoid(model(x).squeeze(1).float()).cpu())
    return torch.cat(out).numpy()

trabajos = [(m, 'r%d_f%d' % (r, f)) for m in MODELOS for r in range(CFG['n_repeats']) for f in range(CFG['n_folds'])]
if LIMITE_CKPT:
    trabajos = [t for t in trabajos if int(t[1][-1]) < LIMITE_CKPT and t[1].startswith('r0')]
filas, control = [], []
t0 = time.time()
X_nu, modelo_tf = None, None
for k, (mname, part) in enumerate(trabajos, 1):
    ck = RES / 'ckpt' / ('%s_%s_best.pt' % (mname, part))
    model = timm.create_model(CFG['models'][mname], pretrained=False, num_classes=1, drop_rate=CFG['drop_rate'])
    model.load_state_dict(torch.load(ck, map_location='cpu'))
    model = model.to(DEV).to(memory_format=torch.channels_last).eval()
    dcm = resolve_data_config({}, model=model)
    tf = T.Compose([T.Resize((CFG['img_size'], CFG['img_size'])), T.ToTensor(), T.Normalize(dcm['mean'], dcm['std'])])
    if modelo_tf != mname:                       # las imágenes se normalizan una vez por arquitectura
        X_nu, modelo_tf = tensores(nu['cache_path'].tolist(), tf), mname
    p_nu = predecir(model, X_nu)
    te = preds_cv[(preds_cv['model'] == mname) & (preds_cv['particion'] == part) & (preds_cv['split'] == 'test')]
    p_te = predecir(model, tensores(te['image_path'].map(cache_de).tolist(), tf))
    dif = float(np.abs(p_te - te['prob'].values).max())
    control.append({'Modelo': mname, 'Partición': part, 'Dif. máx.': dif,
                    'Dif. media': float(np.abs(p_te - te['prob'].values).mean())})
    filas.append(pd.DataFrame({'image_path': nu['image_path'], 'prob': p_nu, 'model': mname, 'particion': part}))
    del model
    rest = (time.time() - t0) / k * (len(trabajos) - k)
    print('%s [%2d/%d] %-15s %s | P(BPR) media en sanas no usadas %.4f | control pliegue de prueba: dif. máx. %.1e | resta ~%.0f s'
          % (time.strftime('%H:%M:%S'), k, len(trabajos), mname, part, p_nu.mean(), dif, rest), flush=True)
pred_nu = pd.concat(filas, ignore_index=True)
pred_nu.to_csv(OUT / 'pred_sanas_no_usadas.csv', index=False)
control = pd.DataFrame(control)
control.to_csv(OUT / 'auditoria' / 'control_reproduccion.csv', index=False)
print('\nDiferencia entre la predicción repetida y la guardada en la validación cruzada: máxima %.2e | media %.2e' % (
    control['Dif. máx.'].max(), control['Dif. media'].mean()))
# Diferencias del orden de 1e-3 son normales (operaciones TF32 de la GPU, orden de los lotes); 1e-2 o más indica
# que el preprocesamiento o los pesos no coinciden con los del experimento principal.
if control['Dif. máx.'].max() > 1e-2:
    print('AVISO: la reproducción no coincide; revisar preprocesamiento antes de interpretar los resultados.')


## 5. Métricas y comparación

In [ ]:
# 5) Métricas. Cada modelo nunca vio las sanas no usadas, ni las imágenes de su propio pliegue de prueba.
#    - Especificidad en sanas no usadas (umbral 0.5, el de las métricas principales, y umbral de validación)
#    - La misma especificidad en las sanas usadas, pero solo las de su pliegue de prueba (referencia)
#    - AUC "con negativos nuevos": positivos = BPR de su pliegue de prueba; negativos = sanas no usadas
rng = np.random.default_rng(CFG['seed'])
limpias = set(nu.loc[~nu['gemela'], 'image_path'])
d = pred_nu.merge(nu[['image_path', 'fuente', 'gemela', 'group', 'vinculo_dudoso', 'escena_cajon']], on='image_path')
d = d.merge(meta_cv[['model', 'particion', 'thr_val', 'test_auc']], on=['model', 'particion'])
te_all = preds_cv[preds_cv['split'] == 'test'].merge(meta_cv[['model', 'particion', 'thr_val']], on=['model', 'particion'])

filas = []
for (mname, part), g in d.groupby(['model', 'particion']):
    gl = g[~g['gemela']]
    te = te_all[(te_all['model'] == mname) & (te_all['particion'] == part)]
    ts, tb = te[te['label'] == 0], te[te['label'] == 1]
    thr = g['thr_val'].iloc[0]
    y = np.r_[np.ones(len(tb)), np.zeros(len(gl))]
    filas.append({'Modelo': mname, 'Partición': part, 'Repetición': int(part[1]),
                  'Esp. no usadas (0.5)': (gl['prob'] < 0.5).mean(),
                  'FP no usadas (0.5)': int((gl['prob'] >= 0.5).sum()),
                  'Esp. no usadas (umbral val.)': (gl['prob'] < thr).mean(),
                  'Esp. no usadas, fuente A (0.5)': (gl.loc[gl['fuente'] == 'A', 'prob'] < 0.5).mean(),
                  'Esp. no usadas, fuente B (0.5)': (gl.loc[gl['fuente'] == 'B', 'prob'] < 0.5).mean(),
                  'Esp. pliegue de prueba (0.5)': (ts['prob'] < 0.5).mean(),
                  'Esp. pliegue de prueba (umbral val.)': (ts['prob'] < ts['thr_val']).mean(),
                  'AUC pliegue de prueba': g['test_auc'].iloc[0],
                  'AUC con negativos nuevos': roc_auc_score(y, np.r_[tb['prob'].values, gl['prob'].values]),
                  'Esp. incluyendo gemelas (0.5)': (g['prob'] < 0.5).mean(),
                  'Esp. sin vínculos dudosos (0.5)': (gl.loc[~gl['vinculo_dudoso'], 'prob'] < 0.5).mean(),
                  'Esp. escena mano + recipiente (0.5)': (gl.loc[gl['escena_cajon'], 'prob'] < 0.5).mean(),
                  'Esp. resto de escenas (0.5)': (gl.loc[~gl['escena_cajon'], 'prob'] < 0.5).mean()})
por_ckpt = pd.DataFrame(filas)
por_ckpt.to_csv(OUT / 'tablas' / 'metricas_por_modelo_entrenado.csv', index=False)

cols = [c for c in por_ckpt.columns if c not in ('Modelo', 'Partición', 'Repetición', 'FP no usadas (0.5)')]
res = por_ckpt.groupby('Modelo')[cols].agg(['mean', 'std']).loc[[m for m in MODELOS if m in por_ckpt['Modelo'].values]]
tabla = pd.DataFrame(index=res.index)
for c in cols:
    tabla[c] = ['%.3f ± %.3f' % (a, b if b == b else 0) for a, b in zip(res[(c, 'mean')], res[(c, 'std')])]
tabla.insert(0, 'FP no usadas (0.5), media', por_ckpt.groupby('Modelo')['FP no usadas (0.5)'].mean().loc[tabla.index].round(1))
print('Sanas no usadas sin gemelas: %d (fuente A %d, fuente B %d). Media ± DE de los %d modelos de cada arquitectura:' % (
    len(limpias), int(((~nu['gemela']) & (nu['fuente'] == 'A')).sum()), int(((~nu['gemela']) & (nu['fuente'] == 'B')).sum()),
    por_ckpt.groupby('Modelo').size().max()))
display(tabla.T)
tabla.T.to_csv(OUT / 'tablas' / 'resumen_sanas_no_usadas.csv')

# 6) Diferencia de especificidad (no usadas − sanas usadas fuera de pliegue), por repetición, con IC95 por
#    bootstrap de grupos (las fotos agrupadas como la misma mazorca se remuestrean juntas).
B = 2000
sanas_us = folds[folds['label'] == 0][['image_path', 'group']]
dif = []
for mname in [m for m in MODELOS if m in d['model'].values]:
    for r in sorted(por_ckpt['Repetición'].unique()):
        # sanas usadas: cada imagen con la predicción del modelo que no la vio (fuera de pliegue)
        oof = te_all[(te_all['model'] == mname) & (te_all['particion'].str.startswith('r%d_' % r)) & (te_all['label'] == 0)]
        oof = oof.merge(sanas_us, on='image_path', suffixes=('', '_f'))
        # no usadas: proporción de los 5 modelos de la repetición que aciertan en cada imagen
        nr = d[(d['model'] == mname) & (d['particion'].str.startswith('r%d_' % r)) & (~d['gemela'])]
        acierto_nu = nr.assign(ok=nr['prob'] < 0.5).groupby(['image_path', 'group'])['ok'].mean().reset_index()
        a_ok = oof.assign(ok=(oof['prob'] < 0.5).astype(float)).groupby('group')['ok'].agg(['sum', 'count'])
        b_ok = acierto_nu.groupby('group')['ok'].agg(['sum', 'count'])
        obs = b_ok['sum'].sum() / b_ok['count'].sum() - a_ok['sum'].sum() / a_ok['count'].sum()
        idx_a = rng.integers(0, len(a_ok), size=(B, len(a_ok)))
        idx_b = rng.integers(0, len(b_ok), size=(B, len(b_ok)))
        boot = (b_ok['sum'].values[idx_b].sum(1) / b_ok['count'].values[idx_b].sum(1)
                - a_ok['sum'].values[idx_a].sum(1) / a_ok['count'].values[idx_a].sum(1))
        dif.append({'Modelo': mname, 'Repetición': r,
                    'Esp. sanas usadas (fuera de pliegue)': a_ok['sum'].sum() / a_ok['count'].sum(),
                    'Esp. sanas no usadas': b_ok['sum'].sum() / b_ok['count'].sum(),
                    'Diferencia': obs, 'IC95 inf': np.percentile(boot, 2.5), 'IC95 sup': np.percentile(boot, 97.5)})
dif = pd.DataFrame(dif)
dif.to_csv(OUT / 'tablas' / 'diferencia_especificidad.csv', index=False)
print('Diferencia de especificidad (umbral 0.5), por repetición:')
display(dif.round(4))


## 6. Figuras y exportación

In [ ]:
# 7) Figuras: distribución de P(BPR) y las sanas no usadas que más modelos clasifican como BPR
fig, ax = plt.subplots(figsize=(6, 3.6))
datos, etiquetas = [], []
for m in [m for m in MODELOS if m in d['model'].values]:
    a = te_all[(te_all['model'] == m) & (te_all['label'] == 0)]['prob'].values
    b = d[(d['model'] == m) & (~d['gemela'])]['prob'].values
    datos += [np.clip(a, 1e-6, 1), np.clip(b, 1e-6, 1)]
    etiquetas += [m + '\nusadas', m + '\nno usadas']
ax.boxplot(datos, whis=(1, 99), showfliers=False)
ax.set_xticks(range(1, len(etiquetas) + 1)); ax.set_xticklabels(etiquetas, fontsize=6, rotation=90)
ax.set_yscale('log'); ax.axhline(0.5, color='red', ls='--', lw=0.8); ax.set_ylabel('P(BPR), escala log')
plt.tight_layout(); fig.savefig(OUT / 'figs' / 'prob_sanas_usadas_vs_no_usadas.png', dpi=200); plt.show()

voto = d[~d['gemela']].groupby('image_path').agg(prob_media=('prob', 'mean'), frac_bpr=('prob', lambda s: (s >= 0.5).mean()))
voto = voto.sort_values('prob_media', ascending=False)
voto.to_csv(OUT / 'tablas' / 'sanas_no_usadas_ranking_bpr.csv')
print('Sanas no usadas que la mayoría de los modelos (> 50 %%) clasifica como BPR: %d' % int((voto['frac_bpr'] > 0.5).sum()))
top = voto.head(12)
fig, axes = plt.subplots(4, 3, figsize=(6, 8))
cp = nu.set_index('image_path')['cache_path']
for ax, (p, t) in zip(axes.ravel(), top.iterrows()):
    ax.imshow(Image.open(cp[p])); ax.axis('off')
    ax.set_title('%s\nP media %.2f | %.0f %% de modelos' % (os.path.basename(p), t['prob_media'], 100 * t['frac_bpr']), fontsize=6)
plt.tight_layout(); fig.savefig(OUT / 'figs' / 'sanas_no_usadas_mas_dudosas.png', dpi=200); plt.show()

with pd.ExcelWriter(OUT / 'tablas' / 'evaluacion_sanas_no_usadas.xlsx') as xw:
    tabla.T.to_excel(xw, sheet_name='resumen')
    dif.to_excel(xw, sheet_name='diferencia', index=False)
    por_ckpt.to_excel(xw, sheet_name='por_modelo', index=False)
    tab.to_excel(xw, sheet_name='fuentes')
print('Resultados en', OUT)
